In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Code_Review_Data")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
print(df.shape)
df.head(5)

(54015, 13)


,repo,pr_number,pr_title,commit_sha,file,line,side,diff_hunk,patch,review_comment,reviewer,created_at,label
0,sympy/sympy,29818,[GSOC 2026] Add Vandermonde systems solver,1617adec5ea6522e6de34418757cd2010c963109,sympy/polys/modulargcd.py,662.0,RIGHT,"@@ -630,6 +632,81 @@ def crt_(cp, cq, p, q):\n...","@@ -5,11 +5,13 @@\n from sympy.ntheory.modular...",```python\ndef vandermonde_interp(basis: list[...,oscarbenjamin,2026-07-02T17:09:49Z,Type Checking
1,sympy/sympy,29961,ENH: assumptions: lra_theory: Improve Preproce...,2b7ec6210f04b2b2d62dd0a84ed7ee3f2890c3ee,sympy/logic/tests/test_lra_theory.py,584.0,RIGHT,"@@ -568,9 +561,8 @@ def test_example_from_pape...","@@ -12,7 +12,8 @@\n from sympy.functions.eleme...",Why are the values of the slack variable chang...,SujalKumar06,2026-06-29T16:18:54Z,Logic/Clarification
2,sympy/sympy,29961,ENH: assumptions: lra_theory: Improve Preproce...,2b7ec6210f04b2b2d62dd0a84ed7ee3f2890c3ee,sympy/logic/tests/test_lra_theory.py,584.0,RIGHT,"@@ -568,9 +561,8 @@ def test_example_from_pape...","@@ -12,7 +12,8 @@\n from sympy.functions.eleme...","Before `y` was removed, the solver would firs...",JosephMehdiyev,2026-06-29T16:58:24Z,Testing
3,sympy/sympy,29961,ENH: assumptions: lra_theory: Improve Preproce...,6f25630f577b970ecc7c81896aa96eae2a364c93,sympy/logic/algorithms/lra_theory.py,355.0,RIGHT,"@@ -348,6 +350,9 @@ def from_encoded_cnf(encod...","@@ -103,8 +103,7 @@\n TODO:\n - Handle non-ra...",I think it would be good to elaborate more on ...,TiloRC,2026-07-02T02:24:02Z,Other/General Discussion
4,sympy/sympy,29961,ENH: assumptions: lra_theory: Improve Preproce...,9b8c6add7fbd5d5a29555250bd18cff155433711,sympy/logic/algorithms/lra_theory.py,733.0,RIGHT,"@@ -722,6 +727,59 @@ def _sep_const_terms(expr...","@@ -103,8 +103,7 @@\n TODO:\n - Handle non-ra...",It might be good to reference where in the pap...,TiloRC,2026-07-02T02:25:34Z,Other/General Discussion


In [3]:
!pip -q install transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 36.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
timm 1.0.28 requires torchvision, which is not installed.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 13.3

In [4]:
import os
import random
import joblib
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
print(df.info())

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54015 entries, 0 to 54014
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   repo            54015 non-null  object 
 1   pr_number       54015 non-null  int64  
 2   pr_title        54015 non-null  object 
 3   commit_sha      54015 non-null  object 
 4   file            54015 non-null  object 
 5   line            53469 non-null  float64
 6   side            54015 non-null  object 
 7   diff_hunk       53577 non-null  object 
 8   patch           54015 non-null  object 
 9   review_comment  54015 non-null  object 
 10  reviewer        53939 non-null  object 
 11  created_at      54015 non-null  object 
 12  label           54015 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 5.4+ MB
None

Columns:
Index(['repo', 'pr_number', 'pr_title', 'commit_sha', 'file', 'line', 'side',
       'diff_hunk', 'patch', 'review_comment', 'reviewer', '

In [7]:
df = df[
    [
        "file",
        "diff_hunk",
        "review_comment",
        "label"
    ]
]

df.head()

,file,diff_hunk,review_comment,label
0,sympy/polys/modulargcd.py,"@@ -630,6 +632,81 @@ def crt_(cp, cq, p, q):\n...",```python\ndef vandermonde_interp(basis: list[...,Type Checking
1,sympy/logic/tests/test_lra_theory.py,"@@ -568,9 +561,8 @@ def test_example_from_pape...",Why are the values of the slack variable chang...,Logic/Clarification
2,sympy/logic/tests/test_lra_theory.py,"@@ -568,9 +561,8 @@ def test_example_from_pape...","Before `y` was removed, the solver would firs...",Testing
3,sympy/logic/algorithms/lra_theory.py,"@@ -348,6 +350,9 @@ def from_encoded_cnf(encod...",I think it would be good to elaborate more on ...,Other/General Discussion
4,sympy/logic/algorithms/lra_theory.py,"@@ -722,6 +727,59 @@ def _sep_const_terms(expr...",It might be good to reference where in the pap...,Other/General Discussion


In [8]:
df = df.dropna()

df = df.reset_index(drop=True)

print(df.shape)

(53577, 4)


In [9]:
def build_input(row):

    return (
        f"FILE:\n{row['file']}\n\n"
        f"DIFF:\n{row['diff_hunk']}\n\n"
        f"COMMENT:\n{row['review_comment']}"
    )

df["input_text"] = df.apply(build_input, axis=1)

df[["input_text"]].head()

,input_text
0,FILE:\nsympy/polys/modulargcd.py\n\nDIFF:\n@@ ...
1,FILE:\nsympy/logic/tests/test_lra_theory.py\n\...
2,FILE:\nsympy/logic/tests/test_lra_theory.py\n\...
3,FILE:\nsympy/logic/algorithms/lra_theory.py\n\...
4,FILE:\nsympy/logic/algorithms/lra_theory.py\n\...


In [10]:
label_encoder = LabelEncoder()

df["label_id"] = label_encoder.fit_transform(df["label"])

print("Number of classes:", len(label_encoder.classes_))

print(label_encoder.classes_)

Number of classes: 16
['API Design' 'Acknowledgment/Approval' 'Bug' 'Commit/Fix Reference'
 'Design/Refactor' 'Documentation' 'Error Handling/UX'
 'Logic/Clarification' 'Naming' 'Opinion/Preference'
 'Other/General Discussion' 'Performance' 'Security' 'Style/Formatting'
 'Testing' 'Type Checking']


In [11]:
joblib.dump(
    label_encoder,
    "/content/drive/MyDrive/label_encoder.pkl"
)

['/content/drive/MyDrive/label_encoder.pkl']

In [12]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label_id"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=42
)

print("Training:", train_df.shape)

print("Validation:", valid_df.shape)

print("Testing:", test_df.shape)

Training: (42861, 6)
Validation: (5358, 6)
Testing: (5358, 6)


In [13]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)

class_weights = torch.tensor(weights, dtype=torch.float)

print(class_weights)

tensor([ 7.6537,  2.6160,  1.3151,  8.6413, 13.8083,  1.1938,  0.8358,  1.9300,
         4.2319,  3.8936,  0.1566,  2.3685,  3.2197,  3.9804,  0.2901,  1.4825])


In [14]:
MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [15]:
MAX_LENGTH = 256

train_encodings = tokenizer(
    train_df["input_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

valid_encodings = tokenizer(
    valid_df["input_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    test_df["input_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

In [16]:
class ReviewDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):

        return len(self.labels)

In [17]:
train_dataset = ReviewDataset(
    train_encodings,
    train_df["label_id"].tolist()
)

valid_dataset = ReviewDataset(
    valid_encodings,
    valid_df["label_id"].tolist()
)

test_dataset = ReviewDataset(
    test_encodings,
    test_df["label_id"].tolist()
)

print(len(train_dataset))
print(len(valid_dataset))
print(len(test_dataset))

42861
5358
5358


In [18]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)

model.to(device)

print("Model Loaded Successfully!")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded Successfully!


In [20]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )[2]

    return {
        "accuracy": accuracy,
        "weighted_f1": f1,
        "macro_f1": macro_f1,
        "precision": precision,
        "recall": recall,
    }

In [21]:
import torch.nn as nn

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [22]:
training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/codebert_output",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    report_to="none"
)

In [23]:
early_stop = EarlyStoppingCallback(
    early_stopping_patience=2
)

In [26]:
trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stop]
)

In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1,Precision,Recall
1,2.318300,1.958974,0.483763,0.497708,0.331483,0.565161,0.483763
2,1.800228,1.773499,0.490295,0.500041,0.391244,0.607882,0.490295
3,1.503309,1.697175,0.559164,0.570935,0.439349,0.621318,0.559164


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16074, training_loss=1.8739458848042647, metrics={'train_runtime': 2084.198, 'train_samples_per_second': 61.694, 'train_steps_per_second': 7.712, 'total_flos': 1.6917856728943104e+16, 'train_loss': 1.8739458848042647, 'epoch': 3.0})

In [28]:
results = trainer.evaluate(test_dataset)
results

Training Loss,Validation Loss,Epoch,Accuracy,Weighted F1,Macro F1,Precision,Recall
1.503309,1.776701,3,0.554125,0.567935,0.413071,0.615702,0.554125


{'eval_loss': 1.776700735092163,
 'eval_accuracy': 0.5541246733855917,
 'eval_weighted_f1': 0.5679350511201949,
 'eval_macro_f1': 0.4130711487246266,
 'eval_precision': 0.6157024048136014,
 'eval_recall': 0.5541246733855917}

In [29]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(
    predictions.predictions,
    axis=1
)

true = predictions.label_ids

In [30]:
print(
    classification_report(
        true,
        preds,
        target_names=label_encoder.classes_
    )
)

                          precision    recall  f1-score   support

              API Design       0.26      0.44      0.33        43
 Acknowledgment/Approval       0.39      0.32      0.35       128
                     Bug       0.33      0.42      0.37       254
    Commit/Fix Reference       0.20      0.28      0.23        39
         Design/Refactor       0.17      0.21      0.19        24
           Documentation       0.48      0.52      0.50       281
       Error Handling/UX       0.36      0.57      0.44       401
     Logic/Clarification       0.25      0.30      0.27       174
                  Naming       0.41      0.47      0.44        79
      Opinion/Preference       0.14      0.37      0.20        86
Other/General Discussion       0.74      0.47      0.58      2139
             Performance       0.33      0.54      0.41       142
                Security       0.48      0.75      0.58       104
        Style/Formatting       0.37      0.49      0.42        84
         

In [31]:
cm = confusion_matrix(
    true,
    preds
)

cm

array([[  19,    0,    0,    2,    0,    0,    6,    3,    0,    0,    7,
           0,    0,    1,    5,    0],
       [   0,   41,    4,    1,    1,    5,   11,    4,    0,    4,   24,
           3,    4,    7,   13,    6],
       [   2,    0,  107,    3,    0,    3,   30,    7,    3,    4,   27,
           2,    5,    0,   42,   19],
       [   0,    0,    0,   11,    1,    3,    8,    0,    0,    0,    8,
           3,    1,    0,    3,    1],
       [   0,    1,    1,    3,    5,    0,    2,    1,    1,    3,    4,
           0,    1,    0,    1,    1],
       [   4,    3,    3,    1,    1,  147,   18,    9,    1,    5,   44,
          13,    2,    3,   22,    5],
       [   2,    7,   23,    5,    3,   14,  230,    8,    0,    4,   55,
          10,   10,    5,   20,    5],
       [   3,    5,    6,    0,    2,    3,   16,   52,    2,   15,   38,
           9,    1,    0,   20,    2],
       [   1,    1,    3,    0,    0,    2,    7,    2,   37,    4,   12,
           2,    1,   

In [32]:
trainer.save_model(
    "/content/drive/MyDrive/CodeBERT_CodeReview_Model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/CodeBERT_CodeReview_Model"
)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [33]:
def predict_review(file_name, diff, comment):

    text = f"""
FILE:
{file_name}

DIFF:
{diff}

COMMENT:
{comment}
"""

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(**inputs)

    pred = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    return label_encoder.inverse_transform([pred])[0]

In [34]:
predict_review(

    "app.py",

    "if user==None:return",

    "Please add proper error handling."
)

'Error Handling/UX'

In [35]:
predict_review(
    "login.py",
    "if password == input:",
    "This condition is incorrect and will always fail."
)

'Bug'